# Factuality Metrics

Aggregates the per-row labels produced by the factuality pipeline
(`factuality_full.csv`) into paper-ready breakdowns.

Verifications covered:
1. **Author** — `author_status` ∈ {`found`, `hallucinated`}
2. **Field** — `field_status`
3. **Seniority** — `seniority_status`
4. **Location** — `location_status`
5. **Ethnicity** — `perceived_ethnicity`

Breakdowns by `model`, `field`, `location`, `language` for each, plus visual
panels (heatmaps, boxplots) and downstream analyses (prestige, long-tail,
inter-model agreement, reproducibility).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', lambda x: f'{x:.1f}')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', context='notebook')

INPUT = Path('../../results/summary/factuality_full.csv')
DIMENSIONS = ['model', 'field', 'location', 'language']

In [ ]:
df = pd.read_csv(INPUT, low_memory=False)
print(f'Rows: {len(df):,}   Columns: {len(df.columns)}')
df[DIMENSIONS + ['author_status', 'field_status', 'seniority_status',
                 'location_status', 'perceived_ethnicity']].head()

## Helpers

In [ ]:
def breakdown(data: pd.DataFrame, status_col: str, dim_col: str) -> pd.DataFrame:
    counts = data.groupby([dim_col, status_col]).size().unstack(fill_value=0)
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    pct = pct.round(1)
    pct['n'] = counts.sum(axis=1)
    return pct.sort_values('n', ascending=False)


def overall(data: pd.DataFrame, status_col: str) -> pd.Series:
    counts = data[status_col].value_counts()
    pct = (counts / counts.sum() * 100).round(1)
    pct.name = f'{status_col} (% over n={len(data):,})'
    return pct


def pct_heatmap(data: pd.DataFrame, row_col: str, col_col: str,
                normalize: str = 'index', title: str = '', cmap: str = 'Blues',
                figsize: tuple = (10, 6)) -> None:
    """Cross-tab + percent-normalised heatmap."""
    ct = pd.crosstab(data[row_col], data[col_col], normalize=normalize) * 100
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(ct, annot=True, fmt='.1f', cmap=cmap, cbar_kws={'label': '%'}, ax=ax)
    ax.set_title(title or f'{row_col} × {col_col}  (row %)')
    ax.set_xlabel(col_col)
    ax.set_ylabel(row_col)
    plt.tight_layout()
    plt.show()

## 1. Author — `found` vs `hallucinated`

In [ ]:
overall(df, 'author_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== author_status × {dim} ===')
    display(breakdown(df, 'author_status', dim))

## 2. Field — match vs mismatch (rows where author was `found`)

In [ ]:
df_field = df[df['author_status'] == 'found'].copy()
print(f'Rows considered: {len(df_field):,}')
overall(df_field, 'field_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== field_status × {dim} ===')
    display(breakdown(df_field, 'field_status', dim))

## 3. Seniority — match vs mismatch

In [ ]:
df_sen = df[df['author_status'] == 'found'].copy()
overall(df_sen, 'seniority_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== seniority_status × {dim} ===')
    display(breakdown(df_sen, 'seniority_status', dim))

## 4. Location — match vs mismatch

In [ ]:
df_loc = df[df['author_status'] == 'found'].copy()
overall(df_loc, 'location_status')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== location_status × {dim} ===')
    display(breakdown(df_loc, 'location_status', dim))

## 5. Ethnicity — distribution

In [ ]:
overall(df, 'perceived_ethnicity')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== perceived_ethnicity × {dim} ===')
    display(breakdown(df, 'perceived_ethnicity', dim))

## 6. Fully correct rows

A row is *fully correct* iff `author_status == 'found'` AND field/seniority/location all match.
Rows with any `unknown` are excluded from the denominator.

In [ ]:
evaluable = df[
    (df['author_status']  == 'found')
    & df['field_status'].isin(['field_match', 'field_mismatch'])
    & df['seniority_status'].isin(['seniority_match', 'seniority_mismatch'])
    & df['location_status'].isin(['location_match', 'location_mismatch'])
].copy()

evaluable['fully_correct'] = (
    (evaluable['field_status']     == 'field_match')
    & (evaluable['seniority_status'] == 'seniority_match')
    & (evaluable['location_status']  == 'location_match')
)

print(f'Evaluable rows: {len(evaluable):,} / {len(df):,}')
print(f'Fully correct: {evaluable["fully_correct"].sum():,}  ({evaluable["fully_correct"].mean()*100:.1f}%)')

In [ ]:
for dim in DIMENSIONS:
    print(f'\n=== fully_correct × {dim} ===')
    g = evaluable.groupby(dim)['fully_correct']
    out = pd.DataFrame({
        'pct_fully_correct': (g.mean() * 100).round(1),
        'n':                 g.size(),
    }).sort_values('n', ascending=False)
    display(out)

## 7. Heatmap — Ethnicity × Location

Para cada país solicitado, qué etnicidades percibidas predominan.

In [ ]:
pct_heatmap(df, row_col='location', col_col='perceived_ethnicity',
            title='Perceived ethnicity by requested country (% per row)',
            cmap='Blues', figsize=(10, 5))

In [ ]:
# Same heatmap, faceted by model
models = sorted(df['model'].dropna().unique())
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5), sharey=True)
if len(models) == 1: axes = [axes]
for ax, m in zip(axes, models):
    sub = df[df['model'] == m]
    ct = pd.crosstab(sub['location'], sub['perceived_ethnicity'], normalize='index') * 100
    sns.heatmap(ct, annot=True, fmt='.0f', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(m); ax.set_xlabel('perceived_ethnicity')
axes[0].set_ylabel('location')
fig.suptitle('Ethnicity × Location, by model (row %)', y=1.02)
plt.tight_layout(); plt.show()

## 8. Heatmap — Requested country × Real country (`oa_country_code`)

Diagonal = match. Off-diagonal = mismatch. Te dice cuándo el modelo recomienda autores
de **otro** país en lugar del solicitado.

In [ ]:
loc_df = df[df['oa_country_code'].notna() & df['location_llm_iso'].notna()].copy()
ct = pd.crosstab(loc_df['location_llm_iso'], loc_df['oa_country_code'], normalize='index') * 100
# Trim to the top-15 destination countries to keep the chart readable
top_dest = ct.sum(axis=0).sort_values(ascending=False).head(15).index
ct = ct[top_dest]
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(ct, annot=True, fmt='.1f', cmap='Reds', cbar_kws={'label': '%'}, ax=ax)
ax.set_title('Requested country (rows) → Real country in OpenAlex (cols)  — top 15 destinations')
plt.tight_layout(); plt.show()

## 9. Career age vs requested seniority

In [ ]:
sen = df[df['seniority_career_age'].notna() & df['seniority_llm_bucket'].notna()].copy()
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=sen, x='model', y='seniority_career_age', hue='seniority_llm_bucket', ax=ax)
ax.axhline(15, color='red', linestyle='--', alpha=0.5, label='Junior/Senior threshold (15y)')
ax.set_ylim(0, 70)
ax.set_title('Real career age (years) vs LLM-requested seniority bucket')
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

In [ ]:
# Median career age table
sen.groupby(['model', 'seniority_llm_bucket'])['seniority_career_age'].agg(['median', 'mean', 'count']).round(1)

## 10. Prestige — h-index, citations, works

¿Los modelos recomiendan superestrellas o investigadores promedio?

In [ ]:
prestige_cols = ['oa_h_index', 'oa_cited_by_count', 'oa_works_count', 'oa_career_age']
df[prestige_cols].describe(percentiles=[.25, .5, .75, .9, .99]).round(1)

In [ ]:
agg = df.groupby('model')[prestige_cols].agg(['median', 'mean']).round(1)
agg

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, col in zip(axes, prestige_cols):
    sns.boxplot(data=df, x='model', y=col, ax=ax, showfliers=False)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 11. Long-tail — most-recommended authors

In [ ]:
author_counts = (df.dropna(subset=['name', 'lastname'])
                   .groupby(['name', 'lastname']).size()
                   .sort_values(ascending=False))
print(f'Unique authors: {len(author_counts):,}')
print(f'Total recommendations: {author_counts.sum():,}')
print(f'Authors appearing only once: {(author_counts == 1).sum():,}  ({(author_counts == 1).mean()*100:.1f}%)')
print(f'Top 1% of authors account for {author_counts.head(int(len(author_counts)*0.01)).sum() / author_counts.sum() * 100:.1f}% of recommendations')
author_counts.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log10(author_counts.values), bins=50, edgecolor='black')
ax.set_xlabel('log10(appearances)')
ax.set_ylabel('# authors')
ax.set_title('Long-tail of author popularity (log scale)')
plt.tight_layout(); plt.show()

## 12. Inter-model agreement (Jaccard)

Para cada par de modelos, qué fracción de autores únicos es compartida.

In [ ]:
model_authors = {
    m: set(zip(sub['name'].fillna(''), sub['lastname'].fillna('')))
    for m, sub in df.dropna(subset=['name', 'lastname']).groupby('model')
}
models = sorted(model_authors)
jacc = pd.DataFrame(index=models, columns=models, dtype=float)
for a in models:
    for b in models:
        sa, sb = model_authors[a], model_authors[b]
        union = sa | sb
        jacc.loc[a, b] = len(sa & sb) / len(union) if union else 0.0
jacc = jacc.astype(float).round(3)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(jacc, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1, ax=ax)
ax.set_title('Inter-model author overlap (Jaccard)')
plt.tight_layout(); plt.show()
jacc

## 13. Reproducibility between runs (same prompt, different `run_id`)

Para cada combinación de prompt (model, role, task, target, field, location, language),
agrupamos los autores devueltos por `run_id` y calculamos la similitud media (Jaccard)
entre runs distintos. `≈ 1` → modelo determinista; `≈ 0` → muy variable.

In [ ]:
PROMPT_KEYS = ['model', 'role', 'task', 'target', 'field', 'subfield', 'location', 'language']

def jaccard(a: set, b: set) -> float:
    u = a | b
    return len(a & b) / len(u) if u else 0.0

rep_rows = []
sub = df.dropna(subset=['name', 'lastname', 'run_id']).copy()
sub['__author'] = list(zip(sub['name'], sub['lastname']))

for keys, g in sub.groupby(PROMPT_KEYS, dropna=False):
    runs = {rid: set(rg['__author']) for rid, rg in g.groupby('run_id')}
    if len(runs) < 2:
        continue
    rids = list(runs.keys())
    pair_scores = [
        jaccard(runs[rids[i]], runs[rids[j]])
        for i in range(len(rids))
        for j in range(i + 1, len(rids))
    ]
    rep_rows.append({**dict(zip(PROMPT_KEYS, keys)),
                     'n_runs':     len(rids),
                     'mean_jaccard': sum(pair_scores) / len(pair_scores)})

rep_df = pd.DataFrame(rep_rows)
print(f'Prompt cells with ≥ 2 runs: {len(rep_df):,}')
rep_df.groupby('model')['mean_jaccard'].agg(['mean', 'median', 'count']).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=rep_df, x='model', y='mean_jaccard', ax=ax)
ax.set_ylim(0, 1)
ax.set_title('Reproducibility — mean Jaccard between runs of the same prompt')
plt.tight_layout(); plt.show()

## 14. Figure 2 — Persona-level Performance (Figure 2-style)

In [ ]:

# ── Normalize persona dimensions ─────────────────────────────────────────────
LOCATION_MAP = {
    'Germany': 'Germany', 'Deutschland': 'Germany', 'Alemania': 'Germany',
    'Canada': 'Canada', 'Canadá': 'Canada', 'Kanada': 'Canada',
    'Japan': 'Japan', 'Japón': 'Japan',
    'South Africa': 'South Africa', 'Sudáfrica': 'South Africa', 'Südafrika': 'South Africa',
    'Ecuador': 'Ecuador',
}
ROLE_MAP = {
    'Director/Recruiter':           'Director/Recruiter',
    'Director(a)/Reclutador(a)':    'Director/Recruiter',
    'Direktor(in)/Rekrutierende(r)':'Director/Recruiter',
    'PhD student':                  'PhD Student',
    'Estudiante de doctorado':      'PhD Student',
    'Doktorand(in)':                'PhD Student',
}
LANG_MAP = {'english': 'English', 'german': 'German', 'spanish': 'Spanish'}

df2 = df.copy()
df2['country']    = df2['location'].map(LOCATION_MAP)
df2['role_canon'] = df2['role'].map(ROLE_MAP)
df2['lang_canon'] = df2['language'].map(LANG_MAP)

# ── Per-row binary metrics ────────────────────────────────────────────────────
df2['refused'] = (df2['valid_flag'] == 'refused').astype(float)
df2['valid']   = df2['valid_flag'].isin(['unchanged', 'cleaned', 'fixed_dict']).astype(float)
df2['found']   = (df2['author_status'] == 'found').astype(float)

found2 = df2[df2['author_status'] == 'found'].copy()
found2['field_match']    = (found2['field_status']    == 'field_match').astype(float)
found2['seniority_match']= (found2['seniority_status']== 'seniority_match').astype(float)
found2['location_match'] = (found2['location_status'] == 'location_match').astype(float)

print("df2 rows:", len(df2), "  found2 rows:", len(found2))


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Config ────────────────────────────────────────────────────────────────────
GROUPS = [
    {
        'name':  'Language',
        'col':   'lang_canon',
        'order': ['English', 'German', 'Spanish'],
        'colors':['#C6D9F1', '#5B9BD5', '#1F5FA6'],
    },
    {
        'name':  'Country',
        'col':   'country',
        'order': ['Canada', 'Ecuador', 'Germany', 'Japan', 'South Africa'],
        'colors':['#C6EFCE', '#70AD47', '#375623', '#1E4B1A', '#A9D18E'],
    },
    {
        'name':  'Role',
        'col':   'role_canon',
        'order': ['Director/Recruiter', 'PhD Student'],
        'colors':['#FFE699', '#FF8C00'],
    },
]

METRICS = [
    {'label': 'Refusals↓',        'col': 'refused',          'data': 'all',   'better': 'low'},
    {'label': 'Validity↑',        'col': 'valid',            'data': 'all',   'better': 'high'},
    {'label': 'Factuality↑',      'col': 'found',            'data': 'all',   'better': 'high'},
    {'label': 'Field Match↑',     'col': 'field_match',      'data': 'found', 'better': 'high'},
    {'label': 'Seniority Match↑', 'col': 'seniority_match',  'data': 'found', 'better': 'high'},
    {'label': 'Location Match↑',  'col': 'location_match',   'data': 'found', 'better': 'high'},
]

GROUP_GAP = 0.6
BAR_H     = 0.55

def mean_ci95(series):
    s = series.dropna()
    n = len(s)
    if n == 0:
        return np.nan, np.nan
    m = s.mean()
    ci = 1.96 * np.sqrt(m * (1 - m) / n)
    return m, ci

# ── Build ordered rows ────────────────────────────────────────────────────────
rows = []
y = 0
group_spans = {}
for g in GROUPS:
    y_start = y
    for i, item in enumerate(g['order']):
        rows.append({
            'group':     g['name'],
            'label':     item,
            'y':         y,
            'color':     g['colors'][i % len(g['colors'])],
            'col_group': g['col'],
        })
        y += 1
    group_spans[g['name']] = (y_start, y - 1)
    y += GROUP_GAP

y_max = y - GROUP_GAP

# ── Figure layout ─────────────────────────────────────────────────────────────
n_metrics  = len(METRICS)
left_frac  = 0.22
fig_w      = 3 + n_metrics * 2.4
fig_h      = y_max * 0.58 + 1.2

fig = plt.figure(figsize=(fig_w, fig_h))

axes = []
for i in range(n_metrics):
    left_pos  = left_frac + i * (1 - left_frac) / n_metrics
    width_pos = (1 - left_frac) / n_metrics * 0.88
    ax = fig.add_axes([left_pos, 0.08, width_pos, 0.82])
    axes.append(ax)

# ── Plot bars ─────────────────────────────────────────────────────────────────
for ax_i, (ax, metric) in enumerate(zip(axes, METRICS)):
    # Select correct data source for this metric
    src = df2 if metric['data'] == 'all' else found2

    # Compute stats per row
    vals = []
    for row in rows:
        d = src[src[row['col_group']] == row['label']][metric['col']]
        vals.append(mean_ci95(d))

    # Best value per group
    best_per_group = {}
    for g in GROUPS:
        g_vals = [v[0] for r, v in zip(rows, vals)
                  if r['group'] == g['name'] and not np.isnan(v[0])]
        if g_vals:
            best_per_group[g['name']] = min(g_vals) if metric['better'] == 'low' else max(g_vals)

    for row, (m, ci) in zip(rows, vals):
        y_pos = row['y']
        if np.isnan(m):
            continue

        best    = best_per_group.get(row['group'], np.nan)
        is_best = (not np.isnan(best)) and (abs(m - best) < 1e-9)

        ax.barh(y_pos, m, height=BAR_H, color=row['color'], alpha=0.9, zorder=2)
        ax.errorbar(m, y_pos, xerr=ci, fmt='none', color='#333333',
                    linewidth=0.9, capsize=2, zorder=3)

        txt    = f"{m:.2f}"
        x_text = max(m - ci - 0.01, 0.01) if m > 0.15 else m + ci + 0.01
        ha     = 'right' if m > 0.15 else 'left'
        ax.text(x_text, y_pos, txt, ha=ha, va='center', fontsize=7,
                fontweight='bold' if is_best else 'normal', color='black', zorder=4)

    # Group separator lines
    prev_group = None
    for row in rows:
        if prev_group is not None and row['group'] != prev_group:
            ax.axhline(row['y'] - GROUP_GAP / 2, color='#AAAAAA',
                       linewidth=0.6, linestyle='--', zorder=1)
        prev_group = row['group']

    ax.set_xlim(0, 1)
    ax.set_ylim(-0.5, y_max - GROUP_GAP + 0.5)
    ax.set_yticks([])
    ax.set_xticks([0, 0.5, 1.0])
    ax.tick_params(axis='x', labelsize=7)
    ax.spines[['left', 'right', 'top']].set_visible(False)
    ax.set_title(metric['label'], fontsize=8, fontweight='bold', pad=5)

# ── Left-side labels ──────────────────────────────────────────────────────────
lax = fig.add_axes([0.0, 0.08, left_frac, 0.82])
lax.set_xlim(0, 1)
lax.set_ylim(-0.5, y_max - GROUP_GAP + 0.5)
lax.axis('off')

for row in rows:
    lax.text(0.96, row['y'], row['label'], ha='right', va='center', fontsize=8)

BRACKET_X = 0.30
for g in GROUPS:
    y0, y1  = group_spans[g['name']]
    mid_y   = (y0 + y1) / 2
    lax.text(0.13, mid_y, g['name'], ha='center', va='center',
             fontsize=8.5, fontweight='bold', rotation=90)
    lax.plot([BRACKET_X, BRACKET_X], [y0 - 0.25, y1 + 0.25], color='black', linewidth=1.1)
    lax.plot([BRACKET_X, BRACKET_X + 0.04], [y0 - 0.25, y0 - 0.25], color='black', linewidth=1.1)
    lax.plot([BRACKET_X, BRACKET_X + 0.04], [y1 + 0.25, y1 + 0.25], color='black', linewidth=1.1)

fig.suptitle(
    'Figure 2: Persona-level performance. Mean values (±95% CI) aggregated by language, country, and role.\n'
    'Bold values indicate best-in-group performance (arrows indicate direction preference).',
    fontsize=8.5, fontweight='bold', y=0.995, va='top'
)

out_dir = Path('../../results/figures')
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'figure2_persona_performance.pdf', bbox_inches='tight', dpi=200)
fig.savefig(out_dir / 'figure2_persona_performance.png', bbox_inches='tight', dpi=200)
print("Saved → results/figures/figure2_persona_performance.{pdf,png}")
plt.show()
